# Prediction Sharpening Analysis

How do predictions get sharper through layers? Track entropy
decrease, component contributions, top-k probability evolution,
and sharpening rate.

## Why This Matters

Prediction sharpening analyzes how the model arrives at its output predictions. Tracking prediction formation across layers reveals the computational process — when the model commits to an answer, what changes its mind, and how confident it is.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prediction_sharpening_analysis import (
    sharpening_trajectory, component_sharpening_contribution,
    top_k_probability_evolution, sharpening_rate,
    prediction_sharpening_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Sharpening Trajectory

Track entropy and top probability through layers.

In [ ]:
result = sharpening_trajectory(model, tokens, position=-1)
print(f"Total sharpening: {result['total_sharpening']:.4f}")
print(f"Initial entropy: {result['initial_entropy']:.4f}")
print(f"Final entropy: {result['final_entropy']:.4f}")
for p in result['per_layer']:
    bar = '█' * int(p['top_prob'] * 40)
    print(f"  Layer {p['layer']}: H={p['entropy']:.4f}, "
          f"top_prob={p['top_prob']:.4f} (tok {p['top_token']}) {bar}")

## Component Sharpening Contribution

How much does attention vs MLP sharpen predictions at each layer?

In [ ]:
for layer in range(4):
    result = component_sharpening_contribution(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer}: attn={result['attn_sharpening']:+.4f}, "
          f"mlp={result['mlp_sharpening']:+.4f}, "
          f"total={result['total_sharpening']:+.4f}")

## Top-K Probability Evolution

Track how the final top-k tokens' probabilities evolve.

In [ ]:
result = top_k_probability_evolution(model, tokens, position=-1, top_k=5)
print(f"Tracked tokens: {result['tracked_tokens']}")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: mass={p['top_k_mass']:.4f}")
    for tok, prob in sorted(p['token_probs'].items(), key=lambda x: -x[1]):
        bar = '█' * int(prob * 40)
        print(f"    T{tok}: {prob:.4f} {bar}")

## Sharpening Rate

Which layer transitions sharpen predictions the most?

In [ ]:
result = sharpening_rate(model, tokens, position=-1)
print(f"Fastest sharpening at layer {result['fastest_sharpening_layer']}")
for t in result['per_transition']:
    flag = ' [SHARPENING]' if t['is_sharpening'] else ' [BROADENING]'
    print(f"  {t['layers']}: rate={t['rate']:+.4f}{flag}")

## Sharpening Summary

In [ ]:
result = prediction_sharpening_summary(model, tokens, position=-1)
for k, v in result.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")